# 06 — Empirical Wavelength Calibration

This notebook demonstrates the empirical wavelength calibration workflow using
real Th/Ar/Ne/Hg lamp measurements recorded on the LHD CMOS echelle detector
(2024-03-05).  The goal is a per-order polynomial **λ(x)** that maps detector
pixel → wavelength without relying on exact spectrometer geometry or prism
ray-tracing.

Topics covered:
1. Inspect the calibration table
2. Load and summarise the wavelength solution
3. Per-order fit quality (residuals, RMS)
4. Evaluate `wavelength_at` and `pixel_at`
5. Compare empirical vs theoretical dispersion
6. Render synthetic lines using the empirical solution
7. Overlay calibration points on a synthetic detector frame

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import echelle_optics as eo
from echelle_optics import (
    load_lhd_cmos_calibration,
    load_lhd_cmos_wavelength_solution,
    lhd_cmos_echelle,
    render_echelle_lines,
    render_white_light,
    central_wavelength_nm,
    linear_dispersion_nm_per_px,
)
from echelle_optics.geometry import GeometryMode

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

## 1. Calibration table overview

In [ ]:
cal_lines = load_lhd_cmos_calibration()
print(f"Total calibration lines loaded : {len(cal_lines)}")
print(f"Order indices in file          : {min(l.order_idx for l in cal_lines)}–{max(l.order_idx for l in cal_lines)}")
print(f"Physical orders                : {min(l.physical_order for l in cal_lines)}–{max(l.physical_order for l in cal_lines)}")
print(f"Wavelength range               : {min(l.wavelength_nm for l in cal_lines):.2f}–{max(l.wavelength_nm for l in cal_lines):.2f} nm")

species_counts: dict = {}
for ln in cal_lines:
    species_counts[ln.species] = species_counts.get(ln.species, 0) + 1
print("\nLines per species:")
for sp, cnt in sorted(species_counts.items(), key=lambda x: -x[1]):
    print(f"  {sp:8s} {cnt:3d}")

In [ ]:
# Show the first 10 entries as a table
import pandas as pd
df_all = pd.DataFrame([
    dict(order=l.physical_order, pixel=l.center_pixel,
         wavelength_nm=l.wavelength_nm, species=l.species)
    for l in cal_lines
])
df_all.head(10)

In [ ]:
# Pixel vs wavelength for all calibration lines, coloured by order
fig, ax = plt.subplots(figsize=(9, 4))
sc = ax.scatter(
    df_all["pixel"], df_all["wavelength_nm"],
    c=df_all["order"], cmap="viridis", s=20, alpha=0.8,
)
plt.colorbar(sc, ax=ax, label="Physical order")
ax.set_xlabel("Center pixel (dispersion axis)")
ax.set_ylabel("Wavelength (nm)")
ax.set_title("All calibration lines — pixel vs wavelength")
plt.tight_layout()
plt.show()

## 2. Load and summarise the wavelength solution

In [ ]:
sol = load_lhd_cmos_wavelength_solution(poly_degree=2)
print(sol.summary())

## 3. Per-order fit quality

In [ ]:
orders_sorted = sol.order_list()
rms_values = [sol.fits[m].rms_nm * 1000 for m in orders_sorted]  # in pm
n_pts = [sol.fits[m].n_points for m in orders_sorted]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

ax1.bar(orders_sorted, rms_values, color="steelblue", alpha=0.8)
ax1.set_xlabel("Physical order")
ax1.set_ylabel("RMS residual (pm)")
ax1.set_title("Per-order fit RMS (quadratic λ(x))")
ax1.axhline(100, color="red", ls="--", lw=0.8, label="100 pm")
ax1.legend(fontsize=8)

ax2.bar(orders_sorted, n_pts, color="darkorange", alpha=0.8)
ax2.set_xlabel("Physical order")
ax2.set_ylabel("Number of calibration lines")
ax2.set_title("Lines used per order")

plt.tight_layout()
plt.show()

In [ ]:
# Show per-point residuals for a few selected orders
selected = [30, 36, 44, 50, 56]
selected = [m for m in selected if sol.has_order(m)]

fig, axes = plt.subplots(1, len(selected), figsize=(3 * len(selected), 3.5), sharey=False)

for ax, m in zip(axes, selected):
    fit = sol.fits[m]
    pixels = [p.center_pixel for p in fit.points]
    residuals_pm = fit.residuals_nm * 1000
    species = [p.species for p in fit.points]

    sc = ax.scatter(pixels, residuals_pm, c=range(len(pixels)), cmap="tab10", s=30, zorder=3)
    for x, y, sp in zip(pixels, residuals_pm, species):
        ax.text(x, y + 2, sp, fontsize=6, ha="center", va="bottom", rotation=45)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Residual (pm)" if m == selected[0] else "")
    ax.set_title(f"Order {m}\nRMS={fit.rms_nm*1000:.1f} pm, n={fit.n_points}")

plt.suptitle("Wavelength fit residuals per point", y=1.02)
plt.tight_layout()
plt.show()

## 4. Evaluate `wavelength_at` and `pixel_at`

In [ ]:
m = 44
fit = sol.fits[m]

x_dense = np.linspace(fit.pixel_min - 50, fit.pixel_max + 50, 1000)
lam_fitted = sol.wavelength_at(x_dense, m)

cal_pixels = [p.center_pixel for p in fit.points]
cal_wavs   = [p.wavelength_nm for p in fit.points]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x_dense, lam_fitted, "b-", lw=1.5, label="Quadratic fit λ(x)")
ax.scatter(cal_pixels, cal_wavs, color="red", zorder=5, s=40, label="Calibration points")
for px, wl, pt in zip(cal_pixels, cal_wavs, fit.points):
    ax.annotate(pt.species, (px, wl), xytext=(0, 6), textcoords="offset points",
                fontsize=7, ha="center")
ax.set_xlabel("Pixel (dispersion axis)")
ax.set_ylabel("Wavelength (nm)")
ax.set_title(f"Order {m} — empirical wavelength solution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Demonstrate inversion
lam_test = 520.0
if fit.wavelength_min_nm < lam_test < fit.wavelength_max_nm:
    x_inv = sol.pixel_at(lam_test, m)
    print(f"pixel_at({lam_test} nm, order {m}) = {x_inv:.1f} px")
    print(f"wavelength_at({x_inv:.1f} px, order {m}) = {float(sol.wavelength_at(x_inv, m)):.4f} nm")

## 5. Empirical vs theoretical dispersion comparison

In [ ]:
spec = lhd_cmos_echelle()

x_mid = 1280.0  # detector centre
dx = 10.0

emp_disp, theo_disp = [], []
for m in sol.order_list():
    fit = sol.fits[m]
    if fit.pixel_min < x_mid - dx and fit.pixel_max > x_mid + dx:
        lam_lo = float(sol.wavelength_at(x_mid - dx, m))
        lam_hi = float(sol.wavelength_at(x_mid + dx, m))
        emp = abs(lam_hi - lam_lo) / (2 * dx)  # nm/px (empirical)
    else:
        emp = np.nan
    theo = linear_dispersion_nm_per_px(
        m, spec.grating.grooves_per_mm, spec.beta_deg,
        spec.focal_length_mm, spec.detector.pixel_size_um,
    )
    emp_disp.append(emp)
    theo_disp.append(theo)

orders_arr = np.array(sol.order_list())
emp_arr = np.array(emp_disp)
theo_arr = np.array(theo_disp)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

ax = axes[0]
ax.plot(orders_arr, theo_arr * 1000, "b-o", ms=4, label="Theoretical (Littrow)")
mask = ~np.isnan(emp_arr)
ax.plot(orders_arr[mask], emp_arr[mask] * 1000, "r--s", ms=4, label="Empirical (calibration)")
ax.set_xlabel("Physical order")
ax.set_ylabel("Dispersion (pm/px)")
ax.set_title("Dispersion at detector centre (x = 1280 px)")
ax.legend(fontsize=8)

ax = axes[1]
ratio = emp_arr[mask] / theo_arr[mask]
ax.plot(orders_arr[mask], ratio, "ko-", ms=4)
ax.axhline(1.0, color="gray", ls="--", lw=0.8)
ax.set_xlabel("Physical order")
ax.set_ylabel("Empirical / Theoretical")
ax.set_title("Dispersion ratio")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=1))

plt.tight_layout()
plt.show()

## 6. Render synthetic lines — theory vs empirical

In [ ]:
# A subset of known Ar / Ne lines present in calibration data
demo_lines = [
    (521.82, 1.0),   # ArII (approximate)
    (530.69, 1.0),   # ArII
    (540.57, 1.0),   # ArI
    (550.80, 1.0),   # ArI
    (560.67, 1.0),   # ThI  (in calibration data)
    (572.02, 1.0),   # ThI
    (585.25, 1.0),   # NeI
    (597.55, 1.0),   # NeI
    (614.31, 1.0),   # NeI
    (630.48, 1.0),   # NeI
    (650.65, 1.0),   # NeI
]

orders = [m for m in sol.order_list() if 40 <= m <= 52]
shape = (2160, 2560)

img_theory = render_echelle_lines(
    demo_lines, spec, orders=orders, shape=shape,
    geometry=GeometryMode.MEASURED_LHD_CMOS,
    psf_sigma_px=1.2, psf_sigma_y_px=8.0, color=True,
)
img_empirical = render_echelle_lines(
    demo_lines, spec, orders=orders, shape=shape,
    geometry=GeometryMode.MEASURED_LHD_CMOS,
    wavelength_solution=sol,
    psf_sigma_px=1.2, psf_sigma_y_px=8.0, color=True,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, img, title in zip(axes, [img_theory, img_empirical],
                           ["Theoretical (Littrow)", "Empirical λ(x)"]):
    ax.imshow(np.clip(img, 0, 1), origin="lower", aspect="auto",
              extent=[0, 2560, 0, 2160])
    ax.set_title(title)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
plt.suptitle("Synthetic emission lines — orders 40–52", y=1.01)
plt.tight_layout()
plt.show()

## 7. Overlay calibration points on a synthetic detector frame

In [ ]:
from echelle_optics import load_lhd_cmos_geometry

geom = load_lhd_cmos_geometry()

# White-light frame with empirical wavelength solution (colour mode)
wl_frame = render_white_light(
    spec, orders=orders, shape=shape,
    geometry=GeometryMode.MEASURED_LHD_CMOS,
    wavelength_solution=sol,
    psf_sigma_y_px=10.0, color=True,
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(np.clip(wl_frame, 0, 1), origin="lower", aspect="auto",
          extent=[0, 2560, 0, 2160])

# Overlay calibration points: plot each as a cross at (pixel, trace-y)
species_colors = {
    "ArI": "white", "ArII": "lightyellow",
    "NeI": "cyan", "ThI": "lime",
    "HgI": "magenta", "HgII": "violet",
    "H-a": "red", "H-g": "blue", "H2": "orange",
}

for ln in cal_lines:
    m = ln.physical_order
    if m not in [t.order for t in geom.traces]:
        continue
    if m not in range(40, 53):
        continue
    x_pt = ln.center_pixel
    y_pt = float(geom.y_at(m, x_pt))
    color = species_colors.get(ln.species, "white")
    ax.plot(x_pt, y_pt, marker="+", color=color, ms=8, mew=1.2)

# Legend
from matplotlib.lines import Line2D
handles = [
    Line2D([0], [0], marker="+", color="none", markerfacecolor=c, markeredgecolor=c,
           ms=8, mew=1.2, label=sp)
    for sp, c in species_colors.items()
]
ax.legend(handles=handles, fontsize=7, loc="upper right",
          framealpha=0.5, ncol=3)

ax.set_title("White-light frame (empirical λ) + calibration line positions — orders 40–52")
ax.set_xlabel("x pixel (dispersion)")
ax.set_ylabel("y pixel (cross-dispersion)")
plt.tight_layout()
plt.show()

In [ ]:
# Single-order zoom: wavelength axis along dispersion
m_zoom = 44
if sol.has_order(m_zoom) and geom.traces:
    fit = sol.fits[m_zoom]
    trace = geom.trace_for_order(m_zoom)
    y_center = float(trace.y_at(1280.0))
    y_margin = 60  # pixels either side of order centre

    y0 = max(0, int(y_center) - y_margin)
    y1 = min(2160, int(y_center) + y_margin)

    strip = wl_frame[y0:y1, :, :]

    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.imshow(np.clip(strip, 0, 1), origin="lower", aspect="auto",
              extent=[0, 2560, 0, y1 - y0])

    # Wavelength tick labels
    x_ticks = np.linspace(fit.pixel_min, fit.pixel_max, 8)
    lam_ticks = [float(sol.wavelength_at(x, m_zoom)) for x in x_ticks]
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f"{l:.2f}" for l in lam_ticks], fontsize=8)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_yticks([])

    # Overlay calibration point positions
    for pt in fit.points:
        y_pt = float(trace.y_at(pt.center_pixel)) - y0
        ax.axvline(pt.center_pixel, color="white", lw=0.6, alpha=0.6)
        ax.text(pt.center_pixel, y_margin * 1.5, pt.species, color="white",
                fontsize=6, ha="center", va="bottom", rotation=45)

    ax.set_title(f"Order {m_zoom} strip — empirical wavelength axis")
    plt.tight_layout()
    plt.show()